##### LS factor Calculation using USGS 10 meter DEM

In [1]:
import rioxarray as rxr
import xarray as xr
import numpy as np
from rasterio.enums import Resampling

# Template grid = your filled DEM (meters)
DEM = rxr.open_rasterio("mart_rusle_output/filled_dem_mart.tif", masked=True).squeeze("band", drop=True)

# C may be on a different grid; align to DEM (nearest preserves zeros on water/built)
C  = rxr.open_rasterio("mart_rusle_output/C_from_NDVI_water_built_zero_mart.tif", masked=True).squeeze("band", drop=True)
C_on_dem = C.rio.reproject_match(DEM, resampling=Resampling.nearest)

# Land = 1 where C > 0 (soil exists); else 0 (water/impervious/NoData)
land_mask = xr.where((~np.isnan(C_on_dem)) & (C_on_dem > 0), 1, 0).astype("uint8")
land_mask = land_mask.rio.write_crs(DEM.rio.crs, inplace=True)
land_mask.rio.to_raster("mart_rusle_output/mart_land_mask_on_DEM.tif", compress="LZW")

#### Final Works for all coordinate types DEM, Re-projects and Fill DEM

In [3]:
# ===========================================
# LS factor from a hydrologically conditioned DEM (meters)
# - Auto-reprojects geographic DEMs to UTM 14N @ 10 m
# - Breaches or fills depressions before flow routing
# - Tries D-Infinity SCA -> MFD -> D8 (cells * cellsize) fallback
# - Correct stream-threshold math (km² -> cells) in meter grid
# - LS = (As/22.13)^m * (sin(slope_rad)/0.0896)^1.3, m clamped to [0.2, 0.6]
# - Optional stream/land masking and LS cap
# ===========================================

import os
import numpy as np
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from whitebox.whitebox_tools import WhiteboxTools

# ---------------- USER INPUTS ----------------
dem_path        = r"mart_rusle_output/filled_dem_mart.tif"  # DEM (any CRS). If not pit-free, we'll fix it.
land_mask_path  = r"mart_rusle_output/mart_land_mask_on_DEM.tif"   # binary (1=land/soil, else 0); or None
LS_CAP          = 300.0                                                # cap extreme LS (None to disable)

# Stream masking
APPLY_STREAM_MASK = True
STREAM_THRESH_KM2 = 0.25  # drainage area threshold (km²) for channels (e.g., 0.1–1.0)

# Output folder
dem_abs  = os.path.abspath(dem_path)
work_dir = os.path.dirname(dem_abs) or "."
out_dir  = os.path.join(work_dir, "LS_python")
os.makedirs(out_dir, exist_ok=True)

# ---------------- LOAD / PROJECT DEM ------------------
DEM = rxr.open_rasterio(dem_abs, masked=True).squeeze("band", drop=True)
crs = DEM.rio.crs

# If DEM is in degrees, project to UTM Zone 14N (meters) for central Texas at 10 m
if (crs is None) or getattr(crs, "is_geographic", False):
    target_crs = "EPSG:26914"   # UTM 14N (meters)
    print(f"[i] DEM is geographic ({crs}); reprojecting to {target_crs} at 10 m...")
    DEM = DEM.rio.reproject(
        target_crs,
        resolution=10.0,
        resampling=Resampling.bilinear
    )
    crs = DEM.rio.crs
    dem_proj_path = os.path.join(out_dir, "dem_projected_utm14.tif")
    DEM.rio.to_raster(dem_proj_path, compress="LZW")
    dem_abs = dem_proj_path

xres, yres = DEM.rio.resolution()
cellsize = float(abs(xres))
cell_area = cellsize * cellsize
print(f"[i] DEM ready. CRS={crs.to_string()}, pixel={cellsize:.2f} m")

# ------------- SLOPE (radians) --------------
# slope_rad = atan( sqrt( (dz/dx)^2 + (dz/dy)^2 ) ), with meter-based spacing
Z = DEM.values.astype("float64")
Zp = np.pad(Z, 1, mode="edge")
dz_dy, dz_dx = np.gradient(Zp, cellsize, cellsize)
slope_rad = np.arctan(np.hypot(dz_dx[1:-1, 1:-1], dz_dy[1:-1, 1:-1]))  # radians

SlopeRad = xr.DataArray(slope_rad, coords=DEM.coords, dims=DEM.dims, name="slope_rad")
SlopeRad = SlopeRad.rio.write_crs(crs)
SlopeRad_path = os.path.join(out_dir, "slope_rad.tif")
SlopeRad.rio.to_raster(SlopeRad_path, compress="LZW")
print("[✓] Wrote:", SlopeRad_path)

# --------- WHITEBOXTOOLS SETUP --------------
wbt = WhiteboxTools()
wbt.verbose = True
wbt.work_dir = work_dir  # safe with spaces

# --------- Ensure pit-free DEM (breach preferred) ----------
try:
    breached = os.path.join(out_dir, "dem_breached.tif")
    wbt.breach_depressions(dem_abs, breached)
    dem_abs = breached
    print("[✓] Breached depressions.")
except Exception as e:
    print(f"[!] Breach failed ({e}); trying fill...")
    try:
        filled = os.path.join(out_dir, "dem_filled_wl.tif")
        wbt.fill_depressions_wang_and_liu(dem_abs, filled)
        dem_abs = filled
        print("[✓] Filled depressions (Wang & Liu).")
    except Exception as e2:
        print(f"[!] Fill also failed ({e2}); continuing with original DEM (may affect flow results).")

# ---- Specific Contributing Area (As, m²/m) ----
As, As_source = None, None

# 1) D-Infinity SCA
try:
    print("[i] Attempting D-Infinity SCA...")
    dinf_dir_path = os.path.join(out_dir, "dinf_dir.tif")
    sca_dinf_path = os.path.join(out_dir, "sca_dinf.tif")
    wbt.d_inf_flow_direction(i=dem_abs, output=dinf_dir_path)
    try:
        wbt.d_inf_flow_accumulation(i=dem_abs, output=sca_dinf_path, pntr=dinf_dir_path)
    except Exception:
        wbt.d_inf_flow_accumulation(i=dem_abs, output=sca_dinf_path)
    if os.path.exists(sca_dinf_path):
        As = rxr.open_rasterio(sca_dinf_path, masked=True).squeeze("band", drop=True).rename("As")
        As = As.rio.write_crs(crs)
        As_source = "D-Infinity"
        print("[✓] D-Infinity SCA computed.")
except Exception as e:
    print(f"[!] D-Infinity not available: {e}\n[i] Trying MFD...")

# 2) MFD SCA
if As is None:
    try:
        sca_mfd_path = os.path.join(out_dir, "sca_mfd.tif")
        print("[i] Attempting MFD SCA...")
        wbt.mfd_flow_accumulation(i=dem_abs, output=sca_mfd_path)
        if os.path.exists(sca_mfd_path):
            As = rxr.open_rasterio(sca_mfd_path, masked=True).squeeze("band", drop=True).rename("As")
            As = As.rio.write_crs(crs)
            As_source = "MFD"
            print("[✓] MFD SCA computed.")
    except Exception as e:
        print(f"[!] MFD not available: {e}\n[i] Falling back to D8 cells × cellsize...")

# 3) D8 fallback: As ≈ (FA_cells + 1) * cellsize   (m²/m)
if As is None:
    fa_cells_path = os.path.join(out_dir, "flow_accum_cells_d8_for_sca.tif")
    print("[i] Computing D8 flow accumulation (cells) for SCA fallback...")
    wbt.d8_flow_accumulation(i=dem_abs, output=fa_cells_path, out_type="cells")
    if not os.path.exists(fa_cells_path):
        raise FileNotFoundError(f"D8 flow accumulation not created: {fa_cells_path}")
    FA_cells = rxr.open_rasterio(fa_cells_path, masked=True).squeeze("band", drop=True)
    As = ((FA_cells.astype("float64") + 1.0) * cellsize).rename("As").rio.write_crs(crs)
    As_source = "D8 cells × cellsize"
    print("[✓] Fallback SCA computed from D8 cells × cellsize.")

# Save SCA
As_out = os.path.join(out_dir, "sca.tif")
As.rio.to_raster(As_out, compress="LZW")
print(f"[✓] Wrote SCA ({As_source}):", As_out)

# ------------- m exponent (continuous + clamp) -------------
# beta = (sin(theta)/0.0896) / (3*sin(theta)^0.8 + 0.56)
# m    = beta / (1 + beta), then clamp to [0.2, 0.6]
sin_s = xr.apply_ufunc(np.sin, SlopeRad)
beta  = (sin_s / 0.0896) / (3.0 * (sin_s ** 0.8) + 0.56)
m     = (beta / (1.0 + beta)).rename("m").clip(0.2, 0.6).rio.write_crs(crs)
m_path = os.path.join(out_dir, "m_value.tif")
m.rio.to_raster(m_path, compress="LZW")
print("[✓] Wrote:", m_path)

# ---------------- L and S -------------------
L = ((As / 22.13) ** m).rename("L").rio.write_crs(crs)
L_path = os.path.join(out_dir, "L_factor.tif")
L.rio.to_raster(L_path, compress="LZW")
print("[✓] Wrote:", L_path)

S = ((sin_s / 0.0896) ** 1.3).rename("S").rio.write_crs(crs)
S_path = os.path.join(out_dir, "S_factor.tif")
S.rio.to_raster(S_path, compress="LZW")
print("[✓] Wrote:", S_path)

# ------------------ LS ----------------------
LS = (L * S).rename("LS")

# ---- Optional stream masking (zero LS on channels) ----
if APPLY_STREAM_MASK:
    print("[i] Building stream mask from D8 flow accumulation (cells)...")
    fa_cells_streams_path = os.path.join(out_dir, "flow_accum_cells_d8_for_streams.tif")
    wbt.d8_flow_accumulation(i=dem_abs, output=fa_cells_streams_path, out_type="cells")

    # STREAM_THRESH_KM2 -> cells (meters-based grid)
    thresh_cells = max(1, int((STREAM_THRESH_KM2 * 1_000_000.0) / cell_area))
    print(f"[i] Stream threshold = {STREAM_THRESH_KM2} km² ≈ {thresh_cells} cells")
    streams_path = os.path.join(out_dir, "streams_mask.tif")
    wbt.extract_streams(flow_accum=fa_cells_streams_path, output=streams_path, threshold=thresh_cells)

    if os.path.exists(streams_path):
        Streams = rxr.open_rasterio(streams_path, masked=True).squeeze("band", drop=True)
        Streams = Streams.rio.reproject_match(DEM, resampling=Resampling.nearest)
        stream_pixels = int(np.nansum((Streams.values > 0).astype(np.int32)))
        print(f"[✓] Streams extracted — pixels: {stream_pixels}")
        LS = xr.where(Streams > 0, 0, LS)
    else:
        print("[!] Stream extraction failed or empty; skipping stream mask.")

# ---- Optional cap on extreme LS ----
if LS_CAP is not None:
    LS = xr.where(LS > LS_CAP, LS_CAP, LS)

# ---- Optional land/non-soil mask ----
if land_mask_path:
    lm_abs = os.path.abspath(land_mask_path)
    if os.path.exists(lm_abs):
        LM = rxr.open_rasterio(lm_abs, masked=True).squeeze("band", drop=True)
        LM = LM.rio.reproject_match(DEM, resampling=Resampling.nearest)
        LS = xr.where(LM == 1, LS, 0)
        print(f"[✓] Applied land mask: {lm_abs}")
    else:
        print(f"[!] land_mask_path not found, skipping: {lm_abs}")

# ---- Write LS (float32, compressed) ----
LS = LS.astype("float32").rio.write_crs(crs)
LS_path = os.path.join(out_dir, "LS_factor.tif")
LS.rio.to_raster(LS_path, compress="LZW")
print("[✓] Wrote:", LS_path)

# ------------------ QA ----------------------
def rng(a: xr.DataArray):
    v = a.values
    return float(np.nanmin(v)), float(np.nanmax(v))

def pct(a: xr.DataArray, pcts=(5,25,50,75,95,99)):
    v = a.values.astype("float64")
    v = v[np.isfinite(v)]
    if v.size == 0:
        return {}
    return {int(p): float(np.nanpercentile(v, p)) for p in pcts}

print(f"[i] As source: {As_source}")
print("Ranges — As[m²/m]:", rng(As),
      "  m:", rng(m),
      "  L:", rng(L),
      "  S:", rng(S),
      "  LS:", rng(LS))
print("Percentiles — L:", pct(L), "  S:", pct(S), "  LS:", pct(LS))

[i] DEM is geographic (EPSG:4269); reprojecting to EPSG:26914 at 10 m...
[i] DEM ready. CRS=EPSG:26914, pixel=10.00 m
[✓] Wrote: C:\Users\User\Box\My Box Notes\Water_Quality_Work\watersheds\RUSLE_cal\mart_rusle_output\LS_python\slope_rad.tif
.\whitebox_tools.exe --run="BreachDepressions" --wd="C:\Users\User\Box\My Box Notes\Water_Quality_Work\watersheds\RUSLE_cal\mart_rusle_output" --dem='C:\Users\User\Box\My Box Notes\Water_Quality_Work\watersheds\RUSLE_cal\mart_rusle_output\LS_python\dem_projected_utm14.tif' --output='C:\Users\User\Box\My Box Notes\Water_Quality_Work\watersheds\RUSLE_cal\mart_rusle_output\LS_python\dem_breached.tif' -v --compress_rasters=False

********************************
* Welcome to BreachDepressions *
* Powered by WhiteboxTools     *
* www.whiteboxgeo.com          *
********************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Pro